In [ ]:
import matplotlib.pyplot as plt
import torch
import random

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CifarAlexNet(number_of_classes=10).to(device)

checkpoint = torch.load(
    MODEL_PATH,
    map_location=device,
    weights_only=False,
)

model.load_state_dict(checkpoint["model_state_dict"])

_, _, test_loader = create_data_loaders()

print("Model loaded successfully!")

class_names = [
    "airplane",
    "automobile",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck",
]

model.eval()

with torch.no_grad():

    images, labels = next(iter(test_loader))

    images = images.to(device)
    labels = labels.to(device)

    outputs = model(images)
    predictions = outputs.argmax(dim=1)

    N = 5
    indices = random.sample(range(images.size(0)), N)

    plt.figure(figsize=(15,4))

    for i, idx in enumerate(indices):

        image = images[idx].cpu()

        image = image * 0.5 + 0.5

        image = image.permute(1,2,0)

        plt.subplot(1, N, i+1)
        plt.imshow(image.numpy())
        plt.axis("off")

        actual = class_names[labels[idx].item()]
        predicted = class_names[predictions[idx].item()]

        color = "green" if actual == predicted else "red"

        plt.title(
            f"GT : {actual}\nPred: {predicted}",
            color=color,
            fontsize=10
        )

    plt.tight_layout()
    plt.show()


# check accuracy
correct = (predictions == labels).sum().item()
accuracy = correct / labels.size(0)

print(f"Batch Accuracy: {accuracy*100:.2f}%")